#### Environment Check


In [1]:
import sys
print(sys.executable)


/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/.venv/bin/python


#### Setup


In [2]:
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field


#### Project Paths


In [3]:
cwd = Path.cwd()

if (cwd / "code").exists() and (cwd / "data").exists():
    LAB_DIR = cwd
elif (cwd.parent / "code").exists() and (cwd.parent / "data").exists():
    LAB_DIR = cwd.parent
else:
    LAB_DIR = Path("..").resolve()

CODE_DIR = LAB_DIR / "code"
DATA_DIR = LAB_DIR / "data"
REPORTS_DIR = LAB_DIR / "reports"

REPORTS_DIR.mkdir(exist_ok=True)

str(LAB_DIR)


'/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab'

#### Import Course Helpers


In [4]:
import sys

sys.path.append(str(CODE_DIR))

from ingest import load_faq_data, build_index
from evaluation_utils import calc_total_price, llm_structured_retry, map_progress


#### Load Ground Truth


In [7]:
ground_truth_path = DATA_DIR / "ground_truth-new.csv"

df_ground_truth = pd.read_csv(ground_truth_path)
ground_truth = df_ground_truth.to_dict(orient="records")

len(ground_truth)


565

#### Load FAQ Documents


In [8]:
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

len(documents)


113

#### Load OpenAI Client


In [9]:
PROJECT_ROOT = LAB_DIR.parents[1]
ENV_PATH = PROJECT_ROOT / ".env"
loaded = load_dotenv(ENV_PATH)
print("Env file:", ENV_PATH)
print("Env loaded:", loaded)
openai_client = OpenAI()
MODEL = "gpt-5.4-mini"


Env file: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/.env
Env loaded: True


#### Create Search Tool


In [10]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"},
    )


#### Create Agent Runner


In [11]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model=MODEL),
)


#### Run One Agent Answer


In [12]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

result.last_message


'A good way to begin is to start with:\n\n- the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)\n- the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/)\n- the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)\n\nYou can start any time—the videos and GitHub materials are available, and deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical weekly workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson notebooks/code.\n3. Read the homework instructions on GitHub.\n4. Submit your answers on the course platform before the deadline.\n\nHomework usually follows the lesson flow, but may use a different dataset or slightly different task.'

#### Extract Tool Calls


In [13]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            if message.get("type") == "function_call":
                tool_calls.append({
                    "name": message.get("name"),
                    "arguments": message.get("arguments"),
                })
            continue

        if getattr(message, "type", None) == "function_call":
            tool_calls.append({
                "name": getattr(message, "name", None),
                "arguments": getattr(message, "arguments", None),
            })

    return tool_calls

tool_calls = extract_tool_calls(result.all_messages)

tool_calls


[{'name': 'search',
  'arguments': '{"query":"recommended way to begin the course organize study each week"}'},
 {'name': 'search',
  'arguments': '{"query":"begin the course weekly study recommended organize each week roadmap homework lectures"}'}]

#### Create Agent Answer Record


In [14]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]

agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": original_doc["answer"],
    "tool_calls": json.dumps(tool_calls),
    "cost": getattr(getattr(result, "cost", None), "total_cost", None),
    "document": doc_id,
}

agent_result


{'question': 'What is the recommended way to begin the course and organize my study each week?',
 'answer_agent': 'A good way to begin is to start with:\n\n- the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)\n- the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/)\n- the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)\n\nYou can start any time—the videos and GitHub materials are available, and deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical weekly workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson notebooks/code.\n3. Read the homework instructions on GitHub.\n4. Submit your answers on the course platform before the deadline.\n\nHomework usually follows the lesson flow, but may use a different dataset or slightly different task.',
 'answer_orig': 'Start with the [LLM Zoomcamp docs](https://datatal

#### Create Agent Answer Function


In [15]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])
    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": json.dumps(tool_calls),
        "cost": getattr(getattr(result, "cost", None), "total_cost", None),
        "document": doc_id,
    }

    return answer_record


#### Generate Agent Answers


In [16]:
agent_questions = ground_truth[:50]
# agent_questions = ground_truth[:10]

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, agent_questions, generate_agent_answer)

len(agent_answers)


  0%|          | 0/50 [00:00<?, ?it/s]

50

#### Save Agent Answers


In [17]:
df_agent = pd.DataFrame(agent_answers)

agent_answers_path = DATA_DIR / "agent-answers.csv"
df_agent.to_csv(agent_answers_path, index=False)

agent_answers_path


PosixPath('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/data/agent-answers.csv')

#### Check Agent Cost


In [18]:
df_agent["cost"].sum()


Decimal('0.06481875')

#### Define Agent Judge Output


In [19]:
class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )


#### Agent Judge Instructions


In [20]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3 can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()


#### Create Agent Judge Function


In [21]:
def evaluate_agent_answer(rec, model=MODEL):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage


#### Test Agent Judge


In [22]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval


AgentEvaluation(answer_reasoning='The agent answer matches the key information from the ground truth: deadlines are on the LLM Zoomcamp course management platform, and homework instructions are in the course GitHub repo, with submissions done on the platform. It does not mention the start point docs or the step-by-step workflow, but those are extra details rather than required core content.', answer_score='good', trajectory_reasoning='The search query was relevant to the question and captured the main concepts of deadlines and submitting homework answers. Only one search call was made, which is reasonable for a straightforward FAQ-style question. The tool call supported the final answer.', trajectory_score='good')

#### Create Agent Record Judge Function


In [23]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage


#### Run Agent Judge


In [24]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)


  0%|          | 0/50 [00:00<?, ?it/s]

#### Split Agent Evaluations And Usage


In [25]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

len(agent_evaluations)


50

#### Create Agent Evaluation DataFrame


In [26]:
df_agent_eval = pd.DataFrame(agent_evaluations)

df_agent_eval.head()


,question,document,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
0,Where can I find the deadlines and submit my h...,04919992b3,good,The agent answer matches the key information f...,good,The search query was relevant and included the...
1,What is the recommended way to begin the cours...,04919992b3,good,The agent’s answer matches the ground truth cl...,good,The single search query is relevant to the que...
2,Is there some official enrollment list that my...,977bf7786c,good,The agent’s answer matches the ground truth. I...,good,The search query was relevant to the question ...
3,"I signed up for the LLM Zoomcamp, but I still ...",977bf7786c,good,The agent’s answer matches the ground truth cl...,good,The single search query was relevant and inclu...
4,"Can I start the course at any time, and does t...",04919992b3,good,The agent answer matches the ground truth well...,good,The search query was relevant and contained th...


#### Check Agent Scores


In [27]:
answer_scores = df_agent_eval["answer_score"].value_counts()
trajectory_scores = df_agent_eval["trajectory_score"].value_counts()

answer_scores, trajectory_scores


(answer_score
 good    43
 bad      7
 Name: count, dtype: int64,
 trajectory_score
 good    48
 bad      2
 Name: count, dtype: int64)

#### Calculate Agent Judge Cost


In [28]:
judge_cost = calc_total_price(usages)

judge_cost


0.008786249999999999

#### Save Agent Judge Results


In [29]:
agent_eval_path = DATA_DIR / "agent-evaluations.csv"
df_agent_eval.to_csv(agent_eval_path, index=False)

agent_eval_path


PosixPath('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/data/agent-evaluations.csv')

#### Save Agent Report


In [30]:
report_path = REPORTS_DIR / "agent_evaluation.md"

agent_generation_cost = df_agent["cost"].sum()
answer_good = int((df_agent_eval["answer_score"] == "good").sum())
trajectory_good = int((df_agent_eval["trajectory_score"] == "good").sum())
total_count = len(df_agent_eval)

report_lines = [
    "# Agent Evaluation Metrics",
    "",
    f"- Agent answers generated: {len(df_agent)}",
    f"- Agent generation cost: {agent_generation_cost}",
    f"- Judge cost: {judge_cost}",
    f"- Good answers: {answer_good}/{total_count}",
    f"- Good trajectories: {trajectory_good}/{total_count}",
    f"- Answers file: {agent_answers_path.name}",
    f"- Evaluations file: {agent_eval_path.name}",
]

with open(report_path, "w") as f:
    f.write("\n".join(report_lines))
    f.write("\n")

report_path


PosixPath('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/reports/agent_evaluation.md')